In [1]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"


In [2]:
import os
import torch
from torchvision import transforms
import timm # provides pretrained models
from timm.layers import SwiGLUPacked


model = timm.create_model("hf-hub:paige-ai/Virchow", pretrained=True, mlp_layer=SwiGLUPacked, act_layer=torch.nn.SiLU)
model.to('mps')
# put the model into inference mode; ensure consistent outputs when extracting embeddings
model.eval()

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1280, out_features=3840, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=1280, out_features=1280, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
      (mlp): GluMlp(
        (fc1): Linear(in_features=1280, out_features=6832, bias=True)
        (act): SiLU()
        (drop1): Dropout(p=0.0, inplace=False)
    

In [3]:
torch.cuda.is_available() 

False

In [4]:
torch.mps.is_available()

True

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import AutoImageProcessor, AutoModel
from timm.data.transforms_factory import create_transform
from timm.data import resolve_data_config
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from tqdm import tqdm
import numpy as np

# # -------------------------------
# # Configuration
# # -------------------------------
DATA_DIR = "/Users/jk/Documents/Lab2/visium/python_analysis/cpath/visium_patches/100um"
MODEL_NAME = "paige-ai/Virchow"   # Hugging Face repo
BATCH_SIZE = 32 # number of images processed in parallel
DEVICE = "mps" 

# # -------------------------------
# # Load model & processor
# # -------------------------------
# processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
# # model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
# # model.eval()  # Freeze encoder

# # -------------------------------
# # Image preprocessing
# # -------------------------------
# transform = transforms.Compose(
#     [
#         transforms.Resize(224),
#         transforms.ToTensor(),
#         transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
#     ]
# )
transform = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))

# -------------------------------
# Create datasets
# -------------------------------
dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Extract class names
class_names = dataset.classes
print("Classes:", class_names)

# -------------------------------
# Extract embeddings
# -------------------------------
num_samples = len(dataset)
feature_dim = 1280  # replace with your model's output dimension

features = np.zeros((num_samples, feature_dim), dtype=np.float32)
labels = np.zeros((num_samples,), dtype=np.int64)

start = 0
with torch.no_grad(): # disable gradient computation
    for images, lbls in tqdm(loader):
        images = images.to('mps')
        outputs = model(images)

        # Convert outputs to embeddings tensor
        embeddings = outputs[0] if isinstance(outputs, (list, tuple)) else outputs
        
        # Get CLS token embeddings (Virchow default)
        if embeddings.ndim == 3:  # (batch_size, seq_len, hidden_dim)
            embeddings = embeddings[:, 0, :]  # CLS token

        batch_size = embeddings.shape[0]  # correct integer
        
        
        features[start:start+batch_size] = embeddings.cpu().numpy()
        labels[start:start+batch_size] = lbls.numpy()
        start += batch_size

# -------------------------------
# Split train/validation
# -------------------------------
# from sklearn.model_selection import train_test_split
# X_train, X_val, y_train, y_val = train_test_split(features, labels, test_size=0.2, stratify=labels, random_state=42)
0
# # -------------------------------
# # Train simple classifier
# # -------------------------------
# clf = LogisticRegression(max_iter=500, multi_class='multinomial')
# clf.fit(X_train, y_train)

# # -------------------------------
# # Evaluate
# # -------------------------------
# y_pred = clf.predict(X_val)
# print(classification_report(y_val, y_pred, target_names=class_names))


Classes: ['cluster0', 'cluster1', 'cluster2', 'cluster3', 'cluster4', 'cluster5']


100%|██████████| 3140/3140 [1:49:18<00:00,  2.09s/it]


0

In [24]:
outputs.shape

torch.Size([32, 257, 1280])

In [ ]:
np.save("features/features_virchow_100um.npy", features)
np.save("labels/labels_virchow_100um.npy", labels)

In [7]:
# -------------------------------
# Split train/validation
# -------------------------------
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(features, labels, test_size=0.2, stratify=labels, random_state=42)

# -------------------------------
# Train simple classifier
# -------------------------------
clf = LogisticRegression(max_iter=500, multi_class='multinomial')
clf.fit(X_train, y_train)

# -------------------------------
# Evaluate
# -------------------------------
y_pred = clf.predict(X_val)
print(classification_report(y_val, y_pred, target_names=class_names))

/Users/jk/miniconda3/envs/uni/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

    cluster0       0.55      0.55      0.55      3941
    cluster1       0.50      0.51      0.50      3900
    cluster2       0.57      0.66      0.61      3782
    cluster3       0.48      0.39      0.43      2763
    cluster4       0.56      0.55      0.56      2868
    cluster5       0.62      0.62      0.62      2841

    accuracy                           0.55     20095
   macro avg       0.55      0.55      0.55     20095
weighted avg       0.55      0.55      0.55     20095



/Users/jk/miniconda3/envs/uni/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
# import the metrics class
from sklearn import metrics

cnf_matrix = metrics.confusion_matrix(y_val, y_pred)
cnf_matrix

array([[2183,  513,  315,  266,  155,  509],
       [ 546, 1971,  521,  369,  316,  177],
       [ 185,  382, 2496,  201,  449,   69],
       [ 383,  516,  358, 1067,  223,  216],
       [ 135,  336,  553,  142, 1591,  111],
       [ 517,  189,  106,  175,   83, 1771]])

In [ ]:
# Train on all data
clf_final = LogisticRegression(max_iter=500, multi_class='multinomial')
clf_final.fit(features, labels)

# Save for later
import joblib
joblib.dump(clf_final, "logistic_classifier/niche_classifier_final_virchow_100um.joblib")

/Users/jk/miniconda3/envs/uni/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/jk/miniconda3/envs/uni/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


['niche_classifier_final_virchow_100um.joblib']